# Bilbao Bar Crawl Optimizer
## Cálculos y decisiones del experimento

Este notebook analiza los artefactos producidos por `scripts/`. El pipeline sigue siendo la fuente canónica de los cálculos; aquí se comprueban sus resultados, se derivan métricas comparables y se explican las decisiones del modelado en grafo.

## 1. Pregunta y escenario

**Pregunta:** ¿qué ruta peatonal encontramos para visitar todos los establecimientos del sector? La ruta debe pasar por los 157 establecimientos y la simulación añade 15 minutos en cada uno para representar el tiempo estimado de tomar un vino.

El escenario activo es un recorrido abierto: salida fija desde Plaza Nueva y final libre. Cada establecimiento debe visitarse una vez. La red se modela como un grafo dirigido `G = (V, E)`, donde los nodos representan puntos de la red y las aristas representan conexiones peatonales con un coste de distancia. La matriz de costes mínimos es compartida por todos los algoritmos.

In [1]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
PROCESSED = DATA / 'processed'

def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

venues_geojson = load_json(PROCESSED / 'venues.geojson')
quality = load_json(PROCESSED / 'quality-report.json')
network = load_json(PROCESSED / 'pedestrian-network-report.json')
matrix_report = load_json(PROCESSED / 'distance-matrix-report.json')
optimization = load_json(PROCESSED / 'optimization-results.json')
summary = load_json(PROCESSED / 'optimization-summary.json')
simulations = load_json(PROCESSED / 'optimization-simulations.json')
config = load_json(DATA / 'study-areas' / 'routing-config.json')
venues = venues_geojson['features']

print(f"Establecimientos exportados: {len(venues)}")
print(f"Fuente: {quality['source_file']}")
print(f"Escenario: {config['scenario']} · salida: {config['start']['name']} · final: libre")
print(f"Velocidad: {config['walking_speed_kmh']} km/h · parada: {config['stop_duration_min']} min")

Establecimientos exportados: 157
Fuente: data/raw/venues-osm-map-editado-5.json
Escenario: open_fixed_start_free_end · salida: Plaza Nueva · final: libre
Velocidad: 4.5 km/h · parada: 15 min


## 2. Establecimientos y cobertura del dataset

El dataset procede de OpenStreetMap y se utiliza como una base realista para probar algoritmos. No se interpreta como un inventario exhaustivo de todos los locales del barrio.

In [2]:
categories = Counter(feature['properties'].get('category', 'sin categoría') for feature in venues)
print('Categorías exportadas:')
for category, count in sorted(categories.items(), key=lambda item: (-item[1], item[0])):
    print(f'  {category:18} {count:>3}')

assert len(venues) == quality['exported_features']
assert quality['duplicate_candidates'] == []
print(f"\nComprobación: {len(venues)} features GeoJSON = {quality['exported_features']} exportados")

Categorías exportadas:
  pub                105
  cafetería           36
  bar                 16

Comprobación: 157 features GeoJSON = 157 exportados


## 3. Red peatonal y nodos de acceso

Cada establecimiento recibe un nodo virtual propio proyectado sobre la arista peatonal más cercana. Esta decisión evita que dos establecimientos distintos sean tratados como si ocuparan exactamente el mismo punto del grafo.

In [3]:
access = load_json(PROCESSED / 'venue-access.json')
unique_access_nodes = {item['node'] for item in access}
print(f"Nodos de red: {network['nodes']:,}")
print(f"Aristas de red: {network['edges']:,}")
print(f"Componentes débilmente conexos: {network['weakly_connected_components']}")
print(f"Nodos virtuales de establecimientos: {network['virtual_access_nodes']}")
print(f"Nodos de acceso únicos observados: {len(unique_access_nodes)}")

assert len(access) == len(venues)
assert len(unique_access_nodes) == len(access)
assert network['virtual_access_nodes'] == len(access)
print('Comprobación del modelo de acceso: OK')

Nodos de red: 5,258
Aristas de red: 14,526
Componentes débilmente conexos: 22
Nodos virtuales de establecimientos: 157
Nodos de acceso únicos observados: 157
Comprobación del modelo de acceso: OK


## 4. Matriz de distancias

La matriz contiene los costes mínimos dirigidos entre todos los establecimientos. Los algoritmos no consultan servicios externos: reciben esta misma matriz y, por tanto, se comparan sobre idénticas condiciones.

In [4]:
total_pairs = matrix_report['ordered_pairs']
reachable = matrix_report['reachable_ordered_pairs']
unreachable = matrix_report['unreachable_ordered_pairs']
coverage = reachable / total_pairs * 100
print(f"Pares ordenados: {total_pairs:,}")
print(f"Pares alcanzables: {reachable:,} ({coverage:.2f}%)")
print(f"Pares desconectados: {unreachable}")
print(f"Pares asimétricos: {matrix_report['asymmetric_unordered_pairs']}")

assert matrix_report['venues'] == len(venues)
assert matrix_report['unique_access_nodes'] == len(venues)
assert unreachable == 0
print('Comprobación de cobertura de la matriz: OK')

Pares ordenados: 24,649
Pares alcanzables: 24,649 (100.00%)
Pares desconectados: 0
Pares asimétricos: 0
Comprobación de cobertura de la matriz: OK


## 5. Comparación de algoritmos

Se ordenan las soluciones por distancia caminando. La referencia aleatoria tiene varias ejecuciones reproducibles; los demás métodos tienen una ejecución determinista en este experimento. La mejora se calcula frente a la media de las ejecuciones aleatorias.

In [5]:
summary_rows = sorted(summary['summary'], key=lambda row: row['best_distance_m'])
random_mean = next(row['mean_distance_m'] for row in summary_rows if row['algorithm'] == 'random')

print(f"{'Algoritmo':35} {'Distancia':>12} {'Caminar':>12} {'Mejora':>10}")
print('-' * 73)
for row in summary_rows:
    print(f"{row['algorithm']:35} {row['best_distance_m']/1000:9.2f} km {row['best_walking_time_min']:8.2f} min {row['improvement_vs_random_mean_pct']:8.2f}%")

best = summary_rows[0]
print(f"\nMejor solución encontrada: {best['algorithm']}")
print(f"Distancia: {best['best_distance_m']/1000:.2f} km")
print(f"Mejora frente a la media aleatoria: {best['improvement_vs_random_mean_pct']:.2f}%")

Algoritmo                              Distancia      Caminar     Mejora
-------------------------------------------------------------------------
or_tools                                 7.66 km   102.10 min    86.18%
two_opt                                  8.13 km   108.43 min    85.32%
nearest_neighbor_multi_start             8.79 km   117.23 min    84.13%
nearest_neighbor                         8.91 km   118.76 min    83.92%
random                                  53.43 km   712.46 min     3.53%

Mejor solución encontrada: or_tools
Distancia: 7.66 km
Mejora frente a la media aleatoria: 86.18%


In [6]:
records_by_algorithm = {}
for record in optimization['algorithms']:
    records_by_algorithm.setdefault(record['algorithm'], []).append(record)

print('Estabilidad de la referencia aleatoria:')
random_distances = [record['distance_m'] for record in records_by_algorithm['random']]
print(f"  ejecuciones: {len(random_distances)}")
print(f"  mínimo: {min(random_distances)/1000:.2f} km")
print(f"  máximo: {max(random_distances)/1000:.2f} km")
print(f"  rango: {(max(random_distances)-min(random_distances))/1000:.2f} km")

assert all(record['venue_count'] == len(venues) for record in optimization['algorithms'])
print('Comprobación de conjunto común de establecimientos: OK')

Estabilidad de la referencia aleatoria:
  ejecuciones: 10
  mínimo: 53.43 km
  máximo: 58.40 km
  rango: 4.97 km
Comprobación de conjunto común de establecimientos: OK


## 6. Simulación temporal

La ruta visita todos los establecimientos. La simulación separa el tiempo caminando del tiempo detenido y añade 15 minutos en cada parada para representar el tiempo estimado de tomar un vino. Es una hipótesis configurable: no representa una observación real ni obliga a consumir alcohol.

In [7]:
def format_minutes(value):
    total = round(value)
    return f'{total // 60} h {total % 60} min' if total >= 60 else f'{total} min'

print(f"{'Algoritmo':35} {'Caminar':>12} {'Paradas':>12} {'Total':>12} {'Jornadas':>10}")
print('-' * 85)
for row in sorted(simulations['simulations'], key=lambda item: item['total_duration_min']):
    walking_share = row['walking_time_min'] / row['total_duration_min'] * 100
    print(f"{row['algorithm']:35} {format_minutes(row['walking_time_min']):>12} {format_minutes(row['stop_time_min']):>12} {format_minutes(row['total_duration_min']):>12} {row['estimated_days']:>10}")
    print(f"  └─ tiempo caminando sobre el total: {walking_share:.1f}%")

expected_stop_time = len(venues) * config['stop_duration_min']
assert all(row['stop_time_min'] == expected_stop_time for row in simulations['simulations'])
print(f"\nParadas estimadas: {len(venues)} × {config['stop_duration_min']} min = {expected_stop_time:.0f} min")

Algoritmo                                Caminar      Paradas        Total   Jornadas
-------------------------------------------------------------------------------------
or_tools                              1 h 42 min  39 h 15 min  40 h 57 min          7
  └─ tiempo caminando sobre el total: 4.2%
two_opt                               1 h 48 min  39 h 15 min   41 h 3 min          7
  └─ tiempo caminando sobre el total: 4.4%
nearest_neighbor_multi_start          1 h 57 min  39 h 15 min  41 h 12 min          7
  └─ tiempo caminando sobre el total: 4.7%
nearest_neighbor                      1 h 59 min  39 h 15 min  41 h 14 min          7
  └─ tiempo caminando sobre el total: 4.8%
random                               11 h 52 min  39 h 15 min   51 h 7 min          9
  └─ tiempo caminando sobre el total: 23.2%

Paradas estimadas: 157 × 15 min = 2355 min


## 7. Reproducibilidad e interpretación

Para regenerar los artefactos desde la raíz del repositorio:

```bash
PATH=/opt/homebrew/bin:$PATH .venv/bin/python scripts/build_pedestrian_network.py --source osm-map
PATH=/opt/homebrew/bin:$PATH .venv/bin/python scripts/split_venue_nodes.py
PATH=/opt/homebrew/bin:$PATH .venv/bin/python scripts/build_distance_matrix.py
PATH=/opt/homebrew/bin:$PATH .venv/bin/python scripts/compare_optimizers.py
PATH=/opt/homebrew/bin:$PATH .venv/bin/python scripts/build_route_simulations.py
```

La conclusión del experimento debe formularse como «mejor solución encontrada bajo este modelo y estos datos». El grafo permite comparar algoritmos de forma reproducible; no convierte automáticamente una heurística en una demostración de optimalidad.